**IMPORT LIBRARIES**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

**LOAD DATASET**

In [ ]:
df = pd.read_csv('../data/FuelConsumptionCo2.csv')
df

In [ ]:
df.head()

**EDA**

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

**Simple linear regression**

In [ ]:
Y = df['CO2EMISSIONS']
Y

In [ ]:
df.select_dtypes(include=['number'])

In [ ]:
df = df[['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_COMB', 'CO2EMISSIONS', ]]

In [ ]:
df

In [ ]:
df.sample(frac = 1, random_state=42).reset_index(drop=True)

In [ ]:
df.corrwith(df['CO2EMISSIONS'])

In [ ]:
df.hist()
plt.tight_layout()

In [ ]:
plt.scatter(df['CO2EMISSIONS'],df['FUELCONSUMPTION_COMB'])
plt.xlabel('CO2EMISSIONS')
plt.ylabel('FUELCONSUMPTION_COMB')

In [ ]:
# plt.figure(figsize=(8,5))
plt.scatter(df['CO2EMISSIONS'],df['ENGINESIZE'], s =10)
plt.xlabel('CO2EMISSIONS')
plt.ylabel('ENGINESIZE')
plt.xlim(100,4000)

No linear pattern obviosly <br>
even after xlim the range grows enormoursly hence of no use

In [ ]:
X = df['FUELCONSUMPTION_COMB'].to_numpy()
X

In [ ]:
Y = df['CO2EMISSIONS'].to_numpy()
Y

Seperating the test and train dataset

In [ ]:
X_train = X[ : int(len(X)*0.8)]
X_test = X[int(len(X)*0.8): ]

Y_train = Y[ : int(len(X)*0.8)]
Y_test = Y[int(len(X)*0.8): ]

Normalize data

In [ ]:
X_train = (X_train - np.mean(X_train)) / np.std(X_train)
Y_train = (Y_train - np.mean(Y_train)) / np.std(Y_train)
X_test = (X_test - np.mean(X_test)) / np.std(X_test)
Y_test = (Y_test - np.mean(Y_test)) / np.std(Y_test)

*If in any case you dont want to do full batch SGD you can use this code to get any desired no of values in a batch just addjust the value of the batch_size and you are ready to go*


```python
def batch_loader(arr, batch_size = 2):
    index = 0
    total = len(arr)
    def get_next_batch():
        nonlocal index
        if index >= total:
            return None
        end = min(index + batch_size , total)
        batch = arr[index : end]
        index = end
        return batch
    return get_next_batch
```

I am using full batch SGD so i didnt have to use it...

In [ ]:
def find_wb(w=0.0, b=0.0, lr=0.1, epochs=1000):
    for epoch in range(epochs):
        Y_pred = X_train * w + b
        error = Y_train - Y_pred

        dw = -2 * np.mean(X_train * error)
        db = -2 * np.mean(error)

        w -=  lr * dw
        b -=  lr * db

        # For better gradient tracking i am using MSE 
        Y_pred_total = w * X_train + b
        total_loss = np.mean((Y_train - Y_pred_total) ** 2)
        print(f"Epoch {epoch+1}: w={w:.4f}, b={b:.4f}, loss={total_loss:.4f}")
        

        if total_loss < 0.001:
            break
    return w, b


    
W, B = find_wb()
print("\nFinal w:", W)
print("Final b:", B)

Gradient Explanation for:<br>
dw = -2 * np.mean(X_batch * error) <br>
db = -2 * np.mean(error)

The model is: Y_pred = w * X + b

The loss function is Mean Squared Error (MSE):<br>
    Loss = (1/n) * ? (Y - Y_pred)^2<br>
         = (1/n) * ? (Y - (w * X + b))^2<br>

To minimize this loss, we use gradient descent.
That requires the partial derivatives of the loss
with respect to w and b:

Gradient w.r.t weight (w):<br>
    dL/dw = -2 * mean(X * (Y - Y_pred))

Gradient w.r.t bias (b):<br>
    dL/db = -2 * mean(Y - Y_pred)

In code:
```python
    error = Y_batch - Y_pred
    dw = -2 * np.mean(X_batch * error)
    db = -2 * np.mean(error)
```

These gradients point in the direction of increasing loss,
so we subtract them during the update:<br>
    w = w - lr * dw<br>
    b = b - lr * db<br>


In [ ]:
Yt_pred = W * X_test + B

Calculate RMSE

In [ ]:
rmse = np.sqrt(np.mean((Y_test - Yt_pred) ** 2))
rmse

In [ ]:
plt.scatter(X_test, Y_test, color = 'blue', label = 'Actual data')

plt.plot(X_test, Yt_pred, color = 'red', label = 'Predicted value')

plt.xlabel('X values')
plt.ylabel('Y values')
plt.title('Linear Regression Fit')
plt.legend()
plt.grid(True)
